# 02 — Classical time-series diagnostics

Companion to `src/price_model/models/classical.py`. The textbook lineage I want to demonstrate here:

1. **Stationarity** — ADF test confirms prices are I(1), log-returns are stationary. Justifies modeling returns, not prices.
2. **Autocorrelation structure** — ACF/PACF on log-returns. Mostly flat past lag 0 → why ARIMA's AR/MA terms struggle on liquid mega-caps.
3. **ARIMA(1,0,1) fit + residual diagnostics** — Ljung-Box on residuals, residual ACF. Confirms model adequacy (or lack thereof).
4. **GARCH(1,1) volatility model** — persistence parameter (α+β), fitted-vs-realized vol over time.
5. **GBM via MLE** — per-ticker (μ, σ), annualized. Forward forecast intervals.
6. **Cross-classical comparison** — once `python -m price_model.cli run --experiment classical` has run, pull the IC/Sharpe table from the prediction store.

Requires: `pip install ".[classical]"` (statsmodels + arch).

**Run with**: `jupyter lab notebooks/02_classical_timeseries.ipynb`

## Setup

In [ ]:
from pathlib import Path

import numpy as np
import pandas as pd
import polars as pl
import matplotlib.pyplot as plt

from price_model.data.loaders import load_panel

# statsmodels + arch via the [classical] extras
from statsmodels.tsa.stattools import adfuller
from statsmodels.graphics.tsaplots import plot_acf, plot_pacf
from statsmodels.stats.diagnostic import acorr_ljungbox
from statsmodels.tsa.arima.model import ARIMA
from arch import arch_model

plt.rcParams['figure.figsize'] = (12, 4)
pd.set_option('display.float_format', lambda x: f'{x:.6f}')

# Use AAPL, NVDA, JPM as representative names: big tech + big tech + a financial
EXAMPLES = ['AAPL', 'NVDA', 'JPM']
PRIMARY = 'AAPL'

panel = load_panel(universe='sp500', start='2017-01-01')
panel = panel.sort(['ticker', 'date']).with_columns(
    (pl.col('adj_close').log() - pl.col('adj_close').log().shift(1))
    .over('ticker')
    .alias('log_return')
)
print(f'Panel: {panel.height:,} rows, {panel["ticker"].n_unique()} tickers, '
      f'{panel["date"].min()} → {panel["date"].max()}')

## 1. Stationarity — ADF test

Null hypothesis of the augmented Dickey-Fuller test: series has a unit root (is non-stationary). We expect:

- **Prices** (random walks) → fail to reject → non-stationary → can't model directly with ARMA.
- **Log-returns** (differenced prices) → reject → stationary → ARMA-appropriate.

This is the empirical justification for the `d=0` choice in `ArimaPerTicker`'s default `(1, 0, 1)` order — we've already differenced once by taking log-returns, so no further differencing is needed.

In [ ]:
rows = []
for tick in EXAMPLES:
    sub = panel.filter(pl.col('ticker') == tick).sort('date').drop_nulls('log_return')
    prices = sub['adj_close'].to_numpy()
    returns = sub['log_return'].to_numpy()
    p_stat, p_pval, *_ = adfuller(prices, autolag='AIC')
    r_stat, r_pval, *_ = adfuller(returns, autolag='AIC')
    rows.append({
        'ticker': tick,
        'price_adf_stat': p_stat, 'price_adf_pval': p_pval,
        'price_stationary?': 'yes' if p_pval < 0.05 else 'NO (unit root)',
        'return_adf_stat': r_stat, 'return_adf_pval': r_pval,
        'return_stationary?': 'yes' if r_pval < 0.05 else 'no',
    })
pd.DataFrame(rows).set_index('ticker')

Visual confirmation: prices wander (non-stationary), returns oscillate around 0 with no trend (stationary).

In [ ]:
sub = panel.filter(pl.col('ticker') == PRIMARY).sort('date').drop_nulls('log_return').to_pandas()
fig, axes = plt.subplots(2, 1, figsize=(12, 6), sharex=True)
axes[0].plot(sub['date'], sub['adj_close'], color='steelblue')
axes[0].set_title(f'{PRIMARY} adjusted close (wanders — non-stationary)')
axes[0].set_ylabel('Price ($)')
axes[1].plot(sub['date'], sub['log_return'], color='gray', linewidth=0.6)
axes[1].axhline(0, color='black', linewidth=0.5)
axes[1].set_title(f'{PRIMARY} daily log-return (oscillates around 0 — stationary)')
axes[1].set_ylabel('log return')
plt.tight_layout(); plt.show()

## 2. ACF / PACF — autocorrelation structure of log-returns

The autocorrelation function (ACF) and partial autocorrelation function (PACF) motivate the AR and MA orders in ARIMA. Box-Jenkins rules of thumb:

- ACF cuts off after lag q, PACF tails off → MA(q).
- ACF tails off, PACF cuts off after lag p → AR(p).
- Both tail off → ARMA(p,q).

For liquid US large-caps you should see **almost all bars within the 95% confidence band** — that's the empirical face of the weak-form EMH on this universe. Whatever ARIMA can extract from past returns alone is, by construction, very small.

In [ ]:
sub = panel.filter(pl.col('ticker') == PRIMARY).sort('date').drop_nulls('log_return')
returns = sub['log_return'].to_numpy()

fig, axes = plt.subplots(1, 2, figsize=(14, 4))
plot_acf(returns, lags=30, ax=axes[0])
axes[0].set_title(f'{PRIMARY} log-return ACF (most bars in CI band = no useful lags)')
plot_pacf(returns, lags=30, ax=axes[1], method='ywm')
axes[1].set_title(f'{PRIMARY} log-return PACF')
plt.tight_layout(); plt.show()

## 3. ARIMA(1,0,1) fit + residual diagnostics

Fit one ARIMA(1,0,1) on the primary ticker's log-returns. Check:

- **AR and MA coefficients** — if both near 0 with high p-values, the model isn't finding signal.
- **Ljung-Box test on residuals** — null: residuals are white noise. We *want* high p-values here (failure to reject means our model captured the autocorrelation that was there to capture).
- **Residual ACF** — should be mostly within the CI band post-fit.

In [ ]:
model = ARIMA(returns, order=(1, 0, 1), trend='c')
result = model.fit(method_kwargs={'warn_convergence': False})
print(result.summary())

# Ljung-Box on residuals at lags 10 and 20
lb = acorr_ljungbox(result.resid, lags=[10, 20], return_df=True)
print('\nLjung-Box on residuals (high p-value = residuals are white noise, good):')
print(lb)

fig, ax = plt.subplots(figsize=(12, 3))
plot_acf(result.resid, lags=30, ax=ax)
ax.set_title(f'{PRIMARY} ARIMA(1,0,1) residual ACF (most bars in band = adequate fit)')
plt.tight_layout(); plt.show()

## 4. GARCH(1,1) — volatility clustering & persistence

ARIMA models the *mean* (which, per the previous section, is mostly noise on liquid stocks). GARCH models the *conditional variance*:

$$\sigma_t^2 = \omega + \alpha \, \varepsilon_{t-1}^2 + \beta \, \sigma_{t-1}^2$$

The **persistence** parameter `α + β` measures how long volatility shocks last. Values near 1.0 (which is the norm for equity returns) mean vol regimes are very sticky — a high-vol day predicts another high-vol day. This is the foundation of every vol-targeting strategy.

In [ ]:
am = arch_model(returns * 100.0, p=1, q=1, mean='Zero', vol='GARCH')
gres = am.fit(disp='off', show_warning=False)
print(gres.summary())

params = gres.params
alpha = float(params.get('alpha[1]', 0.0))
beta = float(params.get('beta[1]', 0.0))
print(f'\nPersistence (α + β): {alpha + beta:.4f}')
print(f'  Half-life of a vol shock: '
      f'{-np.log(2) / np.log(alpha + beta):.1f} trading days')

## 5. Fitted GARCH vol vs. realized — does GARCH actually work?

Plot the model's in-sample conditional volatility against a 20-day rolling realized volatility. They should track closely if GARCH is well-specified.

In [ ]:
cond_vol = gres.conditional_volatility / 100.0   # back to log-return scale
dates = sub['date'].to_pandas()
realized_vol = pd.Series(returns).rolling(20).std()

fig, ax = plt.subplots(figsize=(12, 4))
ax.plot(dates, realized_vol, label='20-day realized vol', color='gray', linewidth=0.8, alpha=0.7)
ax.plot(dates, cond_vol, label='GARCH(1,1) conditional vol', color='crimson', linewidth=1.0)
ax.set_title(f'{PRIMARY}: GARCH conditional vol vs 20-day realized')
ax.set_ylabel('Daily log-return std')
ax.legend()
plt.tight_layout(); plt.show()

## 6. GBM via MLE — per-ticker (μ, σ)

Geometric Brownian Motion: $dS_t = \mu S_t dt + \sigma S_t dW_t$. Closed-form MLE on log-returns gives $\hat{\mu} = \text{mean}$, $\hat{\sigma} = \text{std}$. Annualized using $\sqrt{252}$ scaling for $\sigma$ and $252$ for $\mu$ (the lazy-but-standard convention).

In [ ]:
rows = []
for tick in EXAMPLES:
    sub_t = panel.filter(pl.col('ticker') == tick).sort('date').drop_nulls('log_return')
    r = sub_t['log_return'].to_numpy()
    mu, sigma = float(np.mean(r)), float(np.std(r, ddof=1))
    rows.append({
        'ticker': tick,
        'mu_daily': mu,
        'sigma_daily': sigma,
        'mu_annual': mu * 252,
        'sigma_annual': sigma * np.sqrt(252),
        'sharpe_annual': (mu * 252) / (sigma * np.sqrt(252)),
    })
pd.DataFrame(rows).set_index('ticker')

## 7. Forward forecast interval — visualizing the three classical models

5-day-ahead forecast for the primary ticker, side-by-side:

- **GBM**: $(\mu - \sigma^2/2) \cdot h \pm 2\sigma\sqrt{h}$ — flat point, wide interval driven by historical vol.
- **ARIMA(1,0,1)**: point + symmetric interval from forecast variance.
- **GARCH(1,1)**: point = 0 (by design), interval driven by *current* conditional vol — narrower in calm periods, wider after vol spikes.

In [ ]:
H = 5  # horizon

# GBM
mu, sigma = float(np.mean(returns)), float(np.std(returns, ddof=1))
gbm_point = (mu - 0.5 * sigma * sigma) * H
gbm_std = sigma * np.sqrt(H)

# ARIMA
arima_fc = result.get_forecast(steps=H)
arima_mean_path = arima_fc.predicted_mean
arima_var_path = arima_fc.var_pred_mean
arima_point = float(np.sum(arima_mean_path))
arima_std = float(np.sqrt(np.sum(arima_var_path)))

# GARCH
gfc = gres.forecast(horizon=H, reindex=False)
garch_var = float(np.sum(gfc.variance.values[0])) / 1e4
garch_std = np.sqrt(garch_var)

df = pd.DataFrame([
    {'model': 'GBM',          'point': gbm_point,   'lower': gbm_point   - 2 * gbm_std,   'upper': gbm_point   + 2 * gbm_std},
    {'model': 'ARIMA(1,0,1)', 'point': arima_point, 'lower': arima_point - 2 * arima_std, 'upper': arima_point + 2 * arima_std},
    {'model': 'GARCH(1,1)',   'point': 0.0,         'lower': -2 * garch_std,                'upper':  2 * garch_std},
])
print(f'{PRIMARY} — 5-day-ahead forward log-return forecast intervals:')
print(df.set_index('model'))

fig, ax = plt.subplots(figsize=(8, 4))
for i, row in df.iterrows():
    ax.plot([row['lower'], row['upper']], [i, i], linewidth=4, alpha=0.6)
    ax.scatter([row['point']], [i], color='black', zorder=5)
ax.set_yticks(range(len(df)))
ax.set_yticklabels(df['model'])
ax.axvline(0, color='gray', linestyle=':')
ax.set_xlabel('5-day forward log return (± 2σ band)')
ax.set_title(f'{PRIMARY} 5-day forecast — classical models compared')
plt.tight_layout(); plt.show()

## 8. Classical models vs LightGBM — from the prediction store

Once `python -m price_model.cli run --experiment classical` has been run, the store will contain `arima_classical_v1`, `garch_classical_v1`, `gbm_classical_v1` predictions alongside the LightGBM variants. Pull all of them together.

**Expected pattern:**

- `arima_classical_v1` IC ≈ 0 (within ~±0.001) — confirms past returns don't predict future returns.
- `gbm_classical_v1` IC ≈ 0 — historical drift is too noisy.
- `garch_classical_v1` IC = 0 by construction (point prediction is always 0).
- `lightgbm_kaggle_v2` IC ≈ 0.0074 — material signal *over and above* the EMH null.

In [ ]:
from pathlib import Path as _Path

from price_model.data.loaders import load_panel as _load_panel
from price_model.eval.metrics import compare_models
from price_model.features.targets import add_forward_excess_return
from price_model.serving.store import PredictionStore

# CWD-safe store path (works whether notebook is launched from project root or notebooks/)
NB_DIR = _Path.cwd()
PROJECT_ROOT = NB_DIR if (NB_DIR / 'pyproject.toml').exists() else NB_DIR.parent
STORE_PATH = PROJECT_ROOT / 'artifacts' / 'predictions' / 'predictions.duckdb'

store = PredictionStore(STORE_PATH)
preds = store.query(
    "SELECT model_id, prediction_date AS date, ticker, prediction FROM predictions"
)
if preds.height == 0:
    print('No predictions in store yet. Run an experiment first:')
    print('  python -m price_model.cli run --experiment classical')
else:
    # Compute realized targets and join
    realized_panel = _load_panel(universe='sp500', start='2017-01-01')
    realized_panel = add_forward_excess_return(realized_panel, horizon_days=5)
    realized = realized_panel.select('date', 'ticker', pl.col('y').alias('realized'))
    joined = preds.join(realized, on=['date', 'ticker'], how='inner')
    summary = compare_models(joined, horizon_days=5).sort('information_coefficient', descending=True)
    with pl.Config(tbl_width_chars=300, tbl_cols=20):
        print(summary)

## What this notebook demonstrates

End-to-end Box-Jenkins workflow:

1. Tested stationarity (ADF) — justifies modeling returns, not prices.
2. Inspected autocorrelation (ACF/PACF) — saw that liquid mega-caps have essentially no exploitable lag structure in returns.
3. Fit ARIMA and validated with Ljung-Box residual whiteness test.
4. Fit GARCH and measured volatility persistence — typical α+β ≈ 0.95–0.99 for equity returns means vol shocks are sticky.
5. Fit GBM via MLE and translated to annualized (μ, σ, Sharpe).
6. Compared the three classical models' forward forecast intervals.
7. Where applicable, compared classical IC/Sharpe against LightGBM in the same walk-forward harness.

Net result: classical models give the EMH baseline against which the LightGBM model's IC ≈ 0.007 (t-stat 2.5) is genuinely above noise.